# Análise Telemetria Sistema 1 — 06/06/2026

Análise dos sinais coletados durante incubação (22/05 → 06/06).
Objetivo: validar/falsificar tese de edge antes de implementar Adaptive Sizing.

**Strategy version:** v1.0  
**Ativos:** PETR4, VALE3, ITUB4, BBDC4 (IBOV = benchmark, sem trades)  
**Threshold validação:** PF >= 1.2 líquido | WR >= 45% | n >= 15 trades

---
**INSTRUÇÃO:** Rodar todas as células em ordem. Preencher a seção de Decisão ao final.

In [ ]:
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
from datetime import datetime

COST_PCT = 0.10  # round-trip: 0.05% ida + 0.05% volta

# Carregar signals.jsonl
signals_file = Path("../logs/signals.jsonl")
records = []
with open(signals_file, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

df = pd.DataFrame(records)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"])
df = df.sort_values("timestamp_utc").reset_index(drop=True)

# Filtrar apenas ativos B3 (excluir IBOV e EUR/USD que são benchmark)
B3_ASSETS = ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA"]
df_b3 = df[df["symbol"].isin(B3_ASSETS)].copy()

print(f"Total de sinais (todos):   {len(df)}")
print(f"Sinais B3 (4 ativos):      {len(df_b3)}")
print(f"Período: {df['timestamp_utc'].min()} → {df['timestamp_utc'].max()}")
print(f"Símbolos: {df['symbol'].value_counts().to_dict()}")
print(f"Tipos de sinal: {df_b3['signal'].value_counts().to_dict()}")

In [ ]:
# Expandir features + filtrar sinais com outcome_1h preenchido
features_df = pd.json_normalize(df_b3["features"])
df_full = pd.concat([df_b3.drop("features", axis=1), features_df.add_prefix("feat_")], axis=1)

analyzed = df_full[df_full["outcome_1h"].notna() & (df_full["outcome_1h"] != "ERROR")].copy()
analyzed["outcome_1h"] = pd.to_numeric(analyzed["outcome_1h"], errors="coerce")
analyzed = analyzed.dropna(subset=["outcome_1h"])

# Ajustar sinal de SELL (retorno negativo = ganho)
analyzed["pnl_pct"] = analyzed.apply(
    lambda r: r["outcome_1h"] if r["signal"] == "BUY" else -r["outcome_1h"], axis=1
)
analyzed["pnl_liquido"] = analyzed["pnl_pct"] - COST_PCT
analyzed["win"] = analyzed["pnl_liquido"] > 0

print(f"Sinais analisáveis (com outcome_1h): {len(analyzed)}")
print(f"BUY:  {(analyzed['signal']=='BUY').sum()}")
print(f"SELL: {(analyzed['signal']=='SELL').sum()}")

n = len(analyzed)
if n < 8:
    print("\n⚠️  SISTEMA TRAVADO — menos de 8 trades. CRITÉRIO DE INVALIDAÇÃO ATIVADO.")
elif n < 15:
    print("\n⚠️  Amostra insuficiente (< 15 trades). Edge não pode ser avaliado.")
elif n < 30:
    print(f"\n⚠️  Amostra sub-ótima ({n} trades). Edge pode ser declarado apenas 'promissor'.")
else:
    print(f"\n✅  Amostra estatisticamente confortável ({n} trades >= 30).")

In [ ]:
# === MÉTRICAS 1-4: PF, Win Rate, Expectancy ===

def calc_pf(series):
    gains = series[series > 0].sum()
    losses = abs(series[series < 0].sum())
    return gains / losses if losses > 0 else float('inf')

pf_geral = calc_pf(analyzed["pnl_liquido"])
wr_geral = analyzed["win"].mean()
avg_win = analyzed[analyzed["win"]]["pnl_liquido"].mean()
avg_loss = analyzed[~analyzed["win"]]["pnl_liquido"].mean()
expectancy = analyzed["pnl_liquido"].mean()

print("=== MÉTRICAS GERAIS (líquido de 0.10% round-trip) ===")
print(f"Profit Factor:    {pf_geral:.3f}")
print(f"Win Rate:         {wr_geral:.1%}")
print(f"Avg Win:          {avg_win:.4f}%")
print(f"Avg Loss:         {avg_loss:.4f}%")
print(f"Avg Win/Avg Loss: {abs(avg_win/avg_loss):.2f}x")
print(f"Expectancy/trade: {expectancy:.4f}%")
print(f"Total trades:     {len(analyzed)}")

print("\n=== PF POR ATIVO ===")
for asset in B3_ASSETS:
    sub = analyzed[analyzed["symbol"] == asset]
    if len(sub) == 0:
        print(f"{asset}: sem sinais")
        continue
    pf_a = calc_pf(sub["pnl_liquido"])
    wr_a = sub["win"].mean()
    print(f"{asset}: PF={pf_a:.2f} | WR={wr_a:.1%} | n={len(sub)}")

In [ ]:
# === MÉTRICA 2: PF sem melhor trade (anti-outlier) ===

idx_best = analyzed["pnl_liquido"].idxmax()
best_trade = analyzed.loc[idx_best]
analyzed_sem_outlier = analyzed.drop(idx_best)
pf_sem_outlier = calc_pf(analyzed_sem_outlier["pnl_liquido"])

print(f"Melhor trade removido: {best_trade['symbol']} | {best_trade['timestamp_utc']} | pnl={best_trade['pnl_liquido']:.4f}%")
print(f"PF geral:              {pf_geral:.3f}")
print(f"PF sem melhor trade:   {pf_sem_outlier:.3f}")
print(f"Diferença:             {pf_geral - pf_sem_outlier:.3f}")

if pf_sem_outlier < 1.0:
    print("\n⚠️  CRITÉRIO ANTI-OUTLIER FALHOU: PF < 1.0 sem o melhor trade.")
else:
    print(f"\n✅  Anti-outlier OK: PF={pf_sem_outlier:.3f} mesmo sem o melhor trade.")

In [ ]:
# === MÉTRICA 5: Drawdown máximo ===

CAPITAL = 10000.0
analyzed_sorted = analyzed.sort_values("timestamp_utc").copy()
analyzed_sorted["cum_pnl_pct"] = analyzed_sorted["pnl_liquido"].cumsum()
analyzed_sorted["equity"] = CAPITAL * (1 + analyzed_sorted["cum_pnl_pct"] / 100)
analyzed_sorted["peak"] = analyzed_sorted["equity"].cummax()
analyzed_sorted["drawdown_pct"] = (analyzed_sorted["equity"] - analyzed_sorted["peak"]) / analyzed_sorted["peak"] * 100

max_dd = analyzed_sorted["drawdown_pct"].min()
max_dd_brl = abs(max_dd / 100 * CAPITAL)
final_equity = analyzed_sorted["equity"].iloc[-1]

print(f"Capital inicial:  R$ {CAPITAL:.2f}")
print(f"Capital final:    R$ {final_equity:.2f}")
print(f"PnL acumulado:    R$ {final_equity - CAPITAL:.2f} ({(final_equity/CAPITAL-1)*100:.2f}%)")
print(f"Drawdown máximo:  {max_dd:.2f}% = R$ {max_dd_brl:.2f}")

if abs(max_dd) > 8:
    print("\n🚨  CRITÉRIO DE INVALIDAÇÃO: Drawdown > 8% (R$ 800).")
elif abs(max_dd) > 5:
    print("\n⚠️  Drawdown entre 5% e 8% — acima do critério de validação mas abaixo do de invalidação.")
else:
    print(f"\n✅  Drawdown OK ({max_dd:.2f}% <= 5%).")

In [ ]:
# === MÉTRICAS 6-7: Equity curve + Sharpe/Sortino ===

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# Equity curve
ax1 = axes[0]
ax1.plot(analyzed_sorted["timestamp_utc"], analyzed_sorted["equity"], linewidth=2)
ax1.axhline(CAPITAL, color='gray', linestyle='--', alpha=0.5, label='Capital inicial')
ax1.fill_between(analyzed_sorted["timestamp_utc"], analyzed_sorted["equity"], CAPITAL,
                 where=analyzed_sorted["equity"] >= CAPITAL, alpha=0.2, color='green')
ax1.fill_between(analyzed_sorted["timestamp_utc"], analyzed_sorted["equity"], CAPITAL,
                 where=analyzed_sorted["equity"] < CAPITAL, alpha=0.2, color='red')
ax1.set_title("Equity Curve — Sistema 1 (líquido 0.10% round-trip)")
ax1.set_ylabel("Equity (R$)")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Drawdown
ax2 = axes[1]
ax2.fill_between(analyzed_sorted["timestamp_utc"], analyzed_sorted["drawdown_pct"], 0,
                 alpha=0.5, color='red')
ax2.axhline(-5, color='orange', linestyle='--', alpha=0.7, label='Limite validação (5%)')
ax2.axhline(-8, color='red', linestyle='--', alpha=0.7, label='Limite invalidação (8%)')
ax2.set_title("Drawdown (%)")
ax2.set_ylabel("Drawdown (%)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../knowledge-base/equity_curve_06_06.png", dpi=100, bbox_inches='tight')
plt.show()

# Sharpe e Sortino (auxiliares)
daily_returns = analyzed_sorted.set_index("timestamp_utc")["pnl_liquido"].resample("D").sum()
sharpe = daily_returns.mean() / daily_returns.std() * np.sqrt(252) if daily_returns.std() > 0 else 0
downside = daily_returns[daily_returns < 0].std()
sortino = daily_returns.mean() / downside * np.sqrt(252) if downside > 0 else 0

print(f"Sharpe anualizado (auxiliar): {sharpe:.2f}")
print(f"Sortino anualizado (auxiliar): {sortino:.2f}")
print("(Nota: distorce em amostra pequena — informativo apenas)")

In [ ]:
# === MÉTRICA 8: Distribuição por janela de outcome ===

outcome_cols = ["outcome_5min", "outcome_30min", "outcome_1h", "outcome_4h", "outcome_1d"]
print("=== COBERTURA DE OUTCOMES ===")
for col in outcome_cols:
    if col not in df_b3.columns:
        print(f"{col}: coluna ausente")
        continue
    total = len(df_b3)
    filled = df_b3[col].apply(lambda x: x is not None and str(x) != 'None' and not str(x).startswith('ERROR')).sum()
    print(f"{col:15s}: {filled:3d}/{total} ({filled/total*100:.0f}% preenchido)")

print("\n=== PF POR JANELA DE OUTCOME (se disponível) ===")
for col in outcome_cols:
    if col not in df_full.columns:
        continue
    sub = df_full[df_full[col].notna() & (df_full[col] != "ERROR")].copy()
    sub[col] = pd.to_numeric(sub[col], errors="coerce")
    sub = sub.dropna(subset=[col])
    sub["pnl"] = sub.apply(lambda r: r[col] if r["signal"]=="BUY" else -r[col], axis=1) - COST_PCT
    if len(sub) > 0:
        pf = calc_pf(sub["pnl"])
        print(f"{col:15s}: PF={pf:.2f} (n={len(sub)})")

In [ ]:
# === MÉTRICA 9: Classificação de regime do período ===

import yfinance as yf

start = "2026-05-22"
end = "2026-06-06"

print("=== CLASSIFICAÇÃO DE REGIME DO PERÍODO 22/05 → 06/06 ===")
try:
    ibov = yf.download("^BVSP", start=start, end=end, interval="1d", progress=False)
    if len(ibov) > 0:
        ret_periodo = (ibov["Close"].iloc[-1] / ibov["Close"].iloc[0] - 1) * 100
        vol_realizada = ibov["Close"].pct_change().std() * np.sqrt(252) * 100
        print(f"IBOV retorno no período: {ret_periodo:.2f}%")
        print(f"Volatilidade realizada anualizada: {vol_realizada:.1f}%")

        if ret_periodo > 3:
            tendencia = "ALTA"
        elif ret_periodo < -3:
            tendencia = "BAIXA"
        else:
            tendencia = "LATERAL"

        if vol_realizada > 25:
            vol_regime = "ALTA VOLATILIDADE"
        elif vol_realizada < 12:
            vol_regime = "BAIXA VOLATILIDADE"
        else:
            vol_regime = "VOL NORMAL"

        print(f"\nRegime de tendência: {tendencia}")
        print(f"Regime de volatilidade: {vol_regime}")
        print("\n⚠️  Considere se este período foi representativo antes de generalizar conclusões.")
    else:
        print("Sem dados IBOV para o período — preencher manualmente.")
except Exception as e:
    print(f"Erro ao buscar IBOV: {e}")

print("\n=== EVENTOS MACRO DO PERÍODO (preencher manualmente) ===")
print("[ ] Copom: [data e decisão]")
print("[ ] Payroll EUA: [data e resultado]")
print("[ ] Balanços relevantes: [empresas]")
print("[ ] Eventos políticos BR: [descrição]")
print("[ ] Eventos internacionais: [descrição]")

In [ ]:
# === MÉTRICA 10: Distribuição temporal ===

analyzed_sorted["hora"] = analyzed_sorted["timestamp_utc"].dt.hour
analyzed_sorted["dia_semana"] = analyzed_sorted["timestamp_utc"].dt.day_name()
analyzed_sorted["quinzena"] = analyzed_sorted["timestamp_utc"].dt.day.apply(lambda d: "1a" if d <= 15 else "2a")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Por hora do dia (UTC)
hora_pf = analyzed_sorted.groupby("hora")["pnl_liquido"].apply(calc_pf)
axes[0].bar(hora_pf.index, hora_pf.values, color=['green' if v > 1 else 'red' for v in hora_pf.values])
axes[0].axhline(1.0, color='black', linestyle='--', alpha=0.5)
axes[0].set_title("PF por hora (UTC)")
axes[0].set_xlabel("Hora UTC")
axes[0].set_ylabel("Profit Factor")

# Por dia da semana
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
dia_pf = analyzed_sorted.groupby("dia_semana")["pnl_liquido"].apply(calc_pf).reindex(day_order).dropna()
axes[1].bar(range(len(dia_pf)), dia_pf.values, color=['green' if v > 1 else 'red' for v in dia_pf.values])
axes[1].set_xticks(range(len(dia_pf)))
axes[1].set_xticklabels([d[:3] for d in dia_pf.index], rotation=45)
axes[1].axhline(1.0, color='black', linestyle='--', alpha=0.5)
axes[1].set_title("PF por dia da semana")
axes[1].set_ylabel("Profit Factor")

# Por quinzena
quinzena_pf = analyzed_sorted.groupby("quinzena")["pnl_liquido"].apply(calc_pf)
axes[2].bar(quinzena_pf.index, quinzena_pf.values, color=['green' if v > 1 else 'red' for v in quinzena_pf.values])
axes[2].axhline(1.0, color='black', linestyle='--', alpha=0.5)
axes[2].set_title("PF por quinzena")
axes[2].set_ylabel("Profit Factor")

plt.tight_layout()
plt.savefig("../knowledge-base/temporal_analysis_06_06.png", dpi=100, bbox_inches='tight')
plt.show()

print("PF por hora UTC:")
print(hora_pf.to_string())
print("\nPF por dia da semana:")
print(dia_pf.to_string())

In [ ]:
# === MÉTRICA 11: Healthcheck logs ===

from pathlib import Path

hc_log = Path("../logs/healthcheck.log")
scheduler_log = Path("../scheduler.log")

print("=== HEALTHCHECK DISPAROS ===")
if hc_log.exists():
    alerts = [l for l in hc_log.read_text(encoding="utf-8").splitlines() if "ALERTA" in l or "alert" in l.lower()]
    print(f"Total de alertas: {len(alerts)}")
    for a in alerts:
        print(f"  {a}")
else:
    print("healthcheck.log não encontrado — verificar via scheduler.log")

# Contar disparos no scheduler.log
if scheduler_log.exists():
    lines = scheduler_log.read_text(encoding="utf-8", errors="ignore").splitlines()
    hc_alerts = [l for l in lines if "healthcheck" in l.lower() and ("alerta" in l.lower() or "missing" in l.lower() or "alert" in l.lower())]
    print(f"\nAlertas healthcheck em scheduler.log: {len(hc_alerts)}")
    for a in hc_alerts[-10:]:
        print(f"  {a}")

    # Critério de invalidação: >3 disparos
    n_disparos = len(hc_alerts)
    if n_disparos > 3:
        print(f"\n🚨  CRITÉRIO DE INVALIDAÇÃO ATIVADO: {n_disparos} disparos de healthcheck > 3.")
    else:
        print(f"\n✅  Healthcheck OK: {n_disparos} disparos no período (<= 3).")

In [ ]:
# === SUMÁRIO EXECUTIVO — preencher antes da decisão ===

print("=" * 55)
print("SUMÁRIO EXECUTIVO — Sistema 1 | 06/06/2026")
print("=" * 55)

n = len(analyzed)
print(f"\nTrades analisados:    {n}")
print(f"PF geral (líquido):   {pf_geral:.3f}")
print(f"PF sem outlier:       {pf_sem_outlier:.3f}")
print(f"Win Rate:             {wr_geral:.1%}")
print(f"Expectancy/trade:     {expectancy:.4f}%")
print(f"Drawdown máximo:      {max_dd:.2f}%")
print(f"PnL acumulado:        {analyzed_sorted['cum_pnl_pct'].iloc[-1]:.2f}%")

print("\n--- CRITÉRIOS DE VALIDAÇÃO ---")
print(f"PF >= 1.2:            {'✅' if pf_geral >= 1.2 else '❌'} ({pf_geral:.3f})")
print(f"PF sem outlier > 1.0: {'✅' if pf_sem_outlier > 1.0 else '❌'} ({pf_sem_outlier:.3f})")
print(f"WR >= 45%:            {'✅' if wr_geral >= 0.45 else '❌'} ({wr_geral:.1%})")
print(f"Drawdown <= 5%:       {'✅' if abs(max_dd) <= 5 else '❌'} ({max_dd:.2f}%)")
print(f"Trades >= 15:         {'✅' if n >= 15 else '❌'} ({n})")
print(f"Trades >= 30 (robusto):{'✅' if n >= 30 else '⚠️ '} ({n})")

## Decisão Pós-Análise

**Data da análise:** ___  
**Regime do período:** [ALTA / LATERAL / BAIXA] | [VOL NORMAL / ALTA / BAIXA]  
**Eventos macro relevantes:** ___

---

### Checklist de validação (preencher)

- [ ] **PF >= 1.2 líquido:** ___
- [ ] **PF sem melhor trade > 1.0:** ___
- [ ] **Win rate >= 45%:** ___
- [ ] **Edge em >= 2 ativos:** ___ (ativos com PF > 1.0: ___)
- [ ] **Drawdown <= 5%:** ___
- [ ] **Trades >= 15:** ___  |  **>= 30 (robusto):** ___
- [ ] **Healthcheck <= 3 disparos:** ___
- [ ] **Operacionalmente viável:** ___

---

### Decisão (tabela vinculante — edge_thesis.md)

- [ ] **Validado forte** (PF >= 1.5, n >= 30) → Adaptive Sizing + HMM + micro capital real
- [ ] **Validado base** (1.2 <= PF < 1.5, n >= 30) → Estender +30d, sem mudanças
- [ ] **Promissor** (PF >= 1.2, 15 <= n < 30) → Nova rodada 30d, mesma config
- [ ] **Zona cinza** (0.9 <= PF < 1.2) → Reduzir escopo + análise forense
- [ ] **Invalidado** (PF < 0.9) → PIVOT. Pausar Sistema 1.
- [ ] **Operacionalmente inviável** → Arquivar e reescrever do zero

**Próximo passo:** ___